In [ ]:
"""
Data Quality Documentation
Purpose:
  1. Add an outlier flag column to the modelling dataset (City of London)
  2. Generate a data quality summary document for the dissertation

Inputs:
  data/processed/modelling_dataset.csv

Outputs:
  data/processed/modelling_dataset.csv        
  data/processed/data_quality_summary.md      
"""

import pandas as pd
import os

os.makedirs("data/processed", exist_ok=True)

# STEP 1: Load and add outlier flag
print("=" * 60)
print("  Phase 3.5 — Data Quality Documentation")
print("=" * 60)

df = pd.read_csv("data/processed/modelling_dataset.csv")
print(f"\nLoaded: {df.shape[0]} rows × {df.shape[1]} columns")

df["is_outlier_rate_analysis"] = df["borough"] == "City of London"

flagged = df["is_outlier_rate_analysis"].sum()
print(f"\nOutlier flag added: 'is_outlier_rate_analysis'")
print(f"  Flagged rows: {flagged} (City of London — {flagged} months)")

# Save updated dataset
df.to_csv("data/processed/modelling_dataset.csv", index=False)
print(f"  Updated: data/processed/modelling_dataset.csv")

# Recompute key statistics with and without outlier
df_no_col = df[~df["is_outlier_rate_analysis"]]

borough_rate_full = df.groupby("borough")["crime_rate_per_1000"].mean()
borough_rate_clean = df_no_col.groupby("borough")["crime_rate_per_1000"].mean()

print(f"\nCrime rate per 1,000 statistics:")
print(f"  WITH City of London  : mean = {borough_rate_full.mean():.2f}, max = {borough_rate_full.max():.2f}")
print(f"  WITHOUT City of London: mean = {borough_rate_clean.mean():.2f}, max = {borough_rate_clean.max():.2f}")

# STEP 2: Generate data quality summary
print("\n── Generating data quality summary ──────────────────────")

summary = """# Data Quality Summary

This document records data quality findings discovered during exploratory analysis (EDA), and how each issue is addressed before modelling. It is intended for inclusion in the dissertation's *Methodology* and *Limitations* sections.

---

## 1. City of London — Documented Outlier

### Issue
City of London has the **highest crime rate per 1,000 residents** in the dataset (~94 per 1,000 per month), more than 2.5× the second-highest borough (Westminster at ~34). This appears as a severe outlier in rate-based analysis and distorts borough-level scatter plots and trend lines.

### Cause
City of London is a unique geographic area with:
- Approximately **9,000 resident population** (one of the smallest in the UK)
- Approximately **500,000 daytime workforce** (commuters and tourists)
- Crime is committed largely by and against the daytime population, but the rate is calculated against resident population only.

The "crime per 1,000 *resident* population" metric is therefore not comparable with other boroughs for City of London.

### Mitigation
A boolean column `is_outlier_rate_analysis` has been added to the modelling dataset, flagging all 36 City of London rows. The convention is:

- **For count-based modelling** (predicting `crime_count`): all 33 boroughs are retained. Absolute crime counts are valid measurements regardless of resident population.
- **For rate-based analysis or visualisation** (using `crime_rate_per_1000`): City of London is filtered out using `df[~df["is_outlier_rate_analysis"]]`.

This is documented as a known data limitation in the dissertation.

---

## 2. Multicollinearity Among Deprivation Features

### Issue
The correlation heatmap revealed extremely high correlations (r = 0.97 to 0.99) between four features:
- `imd_score`
- `income_deprivation_score`
- `employment_deprivation_score`
- `claimant_count_rate_2023`

These features are essentially measuring the same underlying construct (socioeconomic deprivation) from slightly different angles.

### Implications
- **Linear Regression** becomes unstable with multicollinear features. Coefficient estimates have inflated variance and become difficult to interpret reliably.
- **Tree-based models** (Random Forest, XGBoost) are robust to multicollinearity. They simply select whichever correlated feature provides the best split at each step.

### Mitigation
The strategy differs by model type:

| Model | Feature Set |
|---|---|
| Linear Regression (baseline) | Use **only `imd_score`** as the deprivation indicator (drop the other three) |
| Random Forest, XGBoost | Keep all deprivation features — multicollinearity does not affect tree splits |

This decision will be explicitly justified in the dissertation's *Model Selection* section.

---

## 3. Apparent Contradiction: Count vs Rate Correlations

### Issue
The EDA revealed a seemingly contradictory finding:
- IMD score vs **crime count** : r = **+0.50** (positive, moderate)
- IMD score vs **crime rate per 1,000** : r = **−0.07** (essentially zero)

At first glance this looks like a data error.

### Explanation
This is not a contradiction — it is a meaningful statistical insight:

- **High-deprivation boroughs tend to be more populous and densely populated.** More people generates more absolute crime regardless of deprivation effects.
- **When we control for population by computing a rate**, the effect of deprivation on crime largely disappears.
- This suggests that the apparent link between deprivation and absolute crime count is **largely mediated by population size**, not deprivation itself.

### Implications
This is a critical finding for the project's framing:
- The model can still predict crime count well using deprivation features (because deprivation correlates with population).
- However, claims that deprivation *causes* higher crime should be made cautiously.
- SHAP analysis in Phase 5 will be valuable for disentangling these effects — particularly to see whether deprivation features retain importance once population is included.

---

## 4. Other Documented Limitations (recap from earlier phases)

| Limitation | Source | Impact |
|---|---|---|
| Date range starts March 2023 | UK Police API does not provide data before 2023-03 for the polygon endpoint | Effective dataset spans 36 months instead of 60+ |
| 4 missing crime values | API connection errors during download (Camden ×3, Wandsworth ×1) | Imputed via linear interpolation within borough; <0.4% of dataset |
| Recorded crime only | Police data reflects reported crimes, not actual incidence | Underrepresentation of unreported crime types |
| Annual socioeconomic data expanded to monthly | Source data is annual; expanded by repeating values across months | Socioeconomic features change slowly, so this is a reasonable approximation |
| Spatial resolution | Crime is street-level but socioeconomic is borough-level | Aggregated to borough level for consistency; loses within-borough variation |

---

## Summary Table — Decisions Made

| Decision | Rationale |
|---|---|
| Flag City of London as `is_outlier_rate_analysis = True` | Resident population not representative of crime exposure |
| Use only `imd_score` for linear baseline | Avoid multicollinearity; preserve interpretability |
| Use full feature set for tree-based models | Multicollinearity does not harm tree splits |
| Document deprivation-population mediation | Critical context for SHAP and discussion sections |
| Retain count-based target as primary | Avoids rate calculation issues; modelling on absolute counts |
| Keep crime_rate_per_1000 as secondary metric | Useful for borough comparison and interpretation, with outlier filtered |

"""

# Save the summary
output_path = "data/processed/data_quality_summary.md"
with open(output_path, "w") as f:
    f.write(summary)

print(f"  Saved: {output_path}")
print(f"  Format: Markdown (can be converted to Word for dissertation)")

# Print summary
print("\n" + "=" * 60)
print("  Data Quality Documentation Complete ✓")
print("=" * 60)
print("""
Updates made:
  1. Modelling dataset now includes `is_outlier_rate_analysis` column
  2. Data quality summary saved as markdown for dissertation use

Outlier handling convention:
  - Count-based modelling   : Keep all 33 boroughs
  - Rate-based analysis     : Filter out City of London

Next step: Phase 4 — Model Building
""")

  Phase 3.5 — Data Quality Documentation

Loaded: 1188 rows × 26 columns

Outlier flag added: 'is_outlier_rate_analysis'
  Flagged rows: 36 (City of London — 36 months)
  Updated: data/processed/modelling_dataset.csv

Crime rate per 1,000 statistics:
  WITH City of London  : mean = 12.99, max = 94.71
  WITHOUT City of London: mean = 10.44, max = 34.02

── Generating data quality summary ──────────────────────
  Saved: data/processed/data_quality_summary.md
  Format: Markdown (can be converted to Word for dissertation)

  Data Quality Documentation Complete ✓

Updates made:
  1. Modelling dataset now includes `is_outlier_rate_analysis` column
  2. Data quality summary saved as markdown for dissertation use

Outlier handling convention:
  - Count-based modelling   : Keep all 33 boroughs
  - Rate-based analysis     : Filter out City of London

Next step: Phase 4 — Model Building

